# Solar Filament Segmentation Challenge 2026 -- Pretraining Data Prep (Colab)

Colab counterpart to `pretrain_gong_kaggle.ipynb`. Clones the
`jp-pretraining-data-prep` branch (which now includes a checked-in manifest
at `manifests/gong_pretrain_manifest.csv` -- no need to rebuild it here) and
drives `scripts/pretrain_data/download_and_convert.py` (combined download +
convert + delete, so raw FITS never piles up on disk) -- see
`PRETRAIN_PLAN.md` for the full design and what's already been verified
against the real archive.

**Differences from the Kaggle version, not just a copy:**
- Colab has open internet access by default -- no "Internet" toggle to
  enable, unlike Kaggle.
- Colab's local disk (`/content/`) is just as ephemeral as Kaggle's working
  directory -- everything is lost on a runtime disconnect/reset. There's no
  Kaggle-Dataset-style persistence step here; instead this notebook mounts
  **Google Drive** and writes JPEG output directly there, so progress
  survives a disconnect. Only the raw FITS scratch space (deleted within
  seconds of each file being converted anyway) stays on local `/content/`.
- This may or may not dodge the rate-limiting/throttling observed against
  this archive from other cloud notebook environments -- Colab and Kaggle
  notebooks often share overlapping Google Cloud IP ranges, so the same
  archive-side defenses could still kick in. Worth trying, not guaranteed.
- Uses the checked-in manifest directly rather than rebuilding it -- to
  extend coverage (e.g. past 2017), rebuild and commit an updated manifest
  from `gong_halpha.py manifest` locally, then pull here.

### 0. Mount Google Drive

Everything under `/content/` disappears on disconnect -- JPEG output and the manifest CSV get written directly under Drive instead, so a disconnect only costs the file(s) actively in flight, not the whole run.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/gong_pretrain"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Output will persist under:", DRIVE_DIR)


### 1. Clone the repo

Requires `jp-pretraining-data-prep` to already be pushed to `origin`.

In [ ]:
!git clone -b jp-pretraining-data-prep https://github.com/jprakash-1/Solar-Filament-Segmentation.git
%cd Solar-Filament-Segmentation
!ls


In [ ]:
# re-run-safe: pick up any commits pushed after this runtime started
!git pull origin jp-pretraining-data-prep


### 2. Install dependencies

Only what Stage 1/2 need (`requests`, `beautifulsoup4`, `astropy`, `Pillow`) --
not the full `requirements.txt`, which also pulls in `torch`/`torchvision`.
Colab, like Kaggle, typically preinstalls a GPU-matched PyTorch build;
reinstalling from PyPI risks the same class of problem hit on the `jp-mvp1`
branch (a wheel that's dropped kernel support for the assigned GPU) -- avoid
it here by simply not installing torch in a notebook that has no use for it.

In [ ]:
!pip install -q requests beautifulsoup4 astropy Pillow


### 3. Stage 1 -- use the checked-in manifest (no rebuilding)

`manifests/gong_pretrain_manifest.csv` came in with the `git clone` above --
37,142 rows, all 6 GONG sites, 2011-01-01 through 2017-01-26 (day-stride 3).
Built and deduplicated already (see `gong_halpha.py`'s `build_manifest` and
its git history for that process); no need to re-query the archive's
directory listings here at all, just start downloading from it directly.

To extend coverage later (e.g. 2017 onwards), rerun
`gong_halpha.py manifest` locally, commit the updated
`manifests/gong_pretrain_manifest.csv`, and pull it here -- rebuilding the
manifest from a notebook isn't necessary.

In [ ]:
import pandas as pd
manifest_path = "manifests/gong_pretrain_manifest.csv"
df = pd.read_csv(manifest_path)
print(f"{len(df)} rows -- site counts:")
print(df["site"].value_counts())
print("date range:", df["timestamp"].min(), "to", df["timestamp"].max())


### 4. Stage 1+2 combined -- download, convert to JPEG, delete raw file

Runs per manifest row: download the FITS file -> convert to JPEG -> delete
the raw file, so raw data never accumulates beyond whatever's actively in
flight. Raw scratch space stays local (`/content/`, fast, and the files are
deleted within seconds anyway); JPEG output and the resulting manifest go
straight to Drive.

Resume-safe (skips a row if its JPEG already exists in `--out-dir`) and
incremental (merges with whatever's already in `--out-dir` rather than
overwriting) -- if this cell gets interrupted by a disconnect, just mount
Drive again and rerun it unchanged.

In [ ]:
!python scripts/pretrain_data/download_and_convert.py \
    --manifest manifests/gong_pretrain_manifest.csv \
    --raw-tmp-dir /content/gong_raw_tmp \
    --out-dir /content/drive/MyDrive/gong_pretrain/processed \
    --workers 8 \
    --processes 8 \
    --log-file /content/drive/MyDrive/gong_pretrain/download_convert.log


### 5. Sanity-check the output before trusting it at scale

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

manifest = pd.read_csv("/content/drive/MyDrive/gong_pretrain/processed/manifest.csv")
print(f"{len(manifest)} JPEGs")

sample = manifest.sample(min(6, len(manifest)), random_state=0)
fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
for ax, (_, row) in zip(np.atleast_1d(axes), sample.iterrows()):
    img = np.array(Image.open(row["path"]))
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(row["site"], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


### 6. Next steps

- Everything under `/content/drive/MyDrive/gong_pretrain/` persists in your
  Google Drive independently of this Colab runtime -- no separate "New
  Dataset" upload step like Kaggle needs.
- If this runtime disconnects mid-run, just reconnect, re-mount Drive, and
  rerun the download/convert cell unchanged -- it's resume-safe (skips any
  row whose output JPEG already exists).
- To extend coverage past 2017-01-26: run `gong_halpha.py manifest` locally
  with a later date range, commit the updated
  `manifests/gong_pretrain_manifest.csv`, then `git pull` in this notebook
  and rerun.
- Stage 3 (MAE pretraining) and Stage 4 (fine-tuning handoff) are still plan
  sections only, not code yet -- see `PRETRAIN_PLAN.md` sections 4-5.